# Programme 3 — Élimination de Gauss et décomposition LR

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, section 2.3.

Ce notebook accompagne `gauss_elimination.py` et illustre :

1. L'algorithme de Gauss et la décomposition $PA = LR$
2. Les **trois stratégies de pivot** : aucun, partiel (Spaltenpivotsuche), total
3. La reproduction de l'**Übung 2.21** du script
4. Pourquoi le pivot partiel est devenu le standard en pratique
5. La borne d'erreur en fonction du conditionnement (section 2.3.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gauss_elimination import (
    StrategiePivot,
    decomposition_lr, resoudre, determinant,
    substitution_avant, substitution_arriere,
    systeme_uebung_2_21, comparer_strategies_uebung_2_21,
    matrice_hilbert, tracer_erreur_vs_conditionnement,
    residu_relatif, erreur_relative,
)

## 1. Décomposition LR — l'idée

L'élimination de Gauss transforme $A$ en une matrice triangulaire supérieure $R$ par opérations élémentaires sur les lignes. Si on stocke aussi les multiplicateurs $\ell_{ik}$ utilisés, on obtient en fait deux matrices :

$$A = L \cdot R$$

où $L$ est triangulaire inférieure à diagonale unitaire, et $R$ triangulaire supérieure. C'est la décomposition LR (LU en anglais).

**Avantage majeur :** une fois $L$ et $R$ calculés (coût $\frac{1}{3} n^3$), résoudre $Ax = b$ pour un nouveau $b$ ne coûte plus que $n^2$ (deux substitutions) au lieu de $\frac{1}{3} n^3$.

In [ ]:
A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])

decomp = decomposition_lr(A, StrategiePivot.PARTIEL)
print('L =')
print(decomp.L)
print('\nR =')
print(decomp.R)
print(f'\nMerkvektor p = {decomp.p}')

# Vérification : P @ A = L @ R
P = decomp.matrice_P()
print(f'\n||PA - LR||_∞ = {np.linalg.norm(P @ A - decomp.L @ decomp.R, np.inf):.2e}')

## 2. Pourquoi le pivot ? — Übung 2.21

Le script étudie le système suivant :

$$\begin{cases} 0{,}00035\, x_1 + 1{,}2654\, x_2 = 3{,}5267 \\ 1{,}2547\, x_1 + 1{,}3182\, x_2 = 6{,}8541 \end{cases}$$

Le pivot $a_{11} = 0{,}00035$ est minuscule. Sans permutation de lignes, on calcule $\ell_{21} = 1{,}2547 / 0{,}00035 \approx 3585$, puis $a_{22} - 3585 \cdot 1{,}2654 \approx 1{,}3182 - 4537{,}4 \approx -4536$. **Auslöschung sévère** : on a soustrait deux nombres très différents en magnitude, et toute la précision de $a_{22}$ est perdue.

Avec pivot partiel, on échange les deux lignes d'abord pour avoir $|a_{11}| = 1{,}2547$, et le calcul reste stable.

In [ ]:
res = comparer_strategies_uebung_2_21()
A, b = systeme_uebung_2_21()
x_ref = np.linalg.solve(A, b)

print(f'Solution exacte (numpy) : {x_ref}\n')
print(f'{"stratégie":>10} | {"erreur relative":>16} | {"résidu relatif":>16}')
print('-' * 50)
for strat, r in res.items():
    if 'erreur' in r:
        print(f'{strat:>10} | {"ÉCHEC":>16} | -')
    else:
        print(f'{strat:>10} | {r["erreur_rel"]:>16.2e} | {r["residu_rel"]:>16.2e}')

Sur ce petit système 2×2 en double précision, même la stratégie « aucun pivot » s'en sort honorablement (erreur $\sim 10^{-13}$). Pour vraiment voir le désastre, prenons un pivot encore plus extrême.

## 3. Cas extrême : pivot $\varepsilon = 10^{-18}$

$$\begin{pmatrix} 10^{-18} & 1 \\ 1 & 1 \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} 1 \\ 2 \end{pmatrix}$$

Solution exacte : $x_1 \approx x_2 \approx 1$.

In [ ]:
eps = 1e-18
A = np.array([[eps, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])
x_ref = np.linalg.solve(A, b)

print(f'Solution exacte : {x_ref}\n')
for strat in StrategiePivot:
    x = resoudre(A, b, strat)
    print(f'  {strat.value:>8}  x = {x},  erreur relative = {erreur_relative(x, x_ref):.2e}')

**Sans pivot :** $x_1 = 0$ au lieu de $1$. **100 % d'erreur.** L'algorithme est techniquement applicable mais le résultat est inutilisable.

**Avec pivot (partiel ou total) :** précision machine, $\sim 10^{-16}$. 

C'est exactement la mise en garde de la section 2.3.5 du script : *« Wenn $a_{kk}^{(k-1)}$ einfach so betragsmäßig verhältnismäßig klein ist, können hieraus Stabilitätsprobleme resultieren. »*

## 4. Substitutions avant et arrière (formule 2.10)

Une fois $PA = LR$ obtenue, résoudre $Ax = b$ se fait en trois temps :

1. Permuter : $b' = Pb$
2. Substitution avant : résoudre $Ly = b'$
3. Substitution arrière (formule 2.10) : résoudre $Rx = y$ par $x_i = \frac{1}{r_{ii}}\left(c_i - \sum_{j>i} r_{ij} x_j\right)$

In [ ]:
rng = np.random.default_rng(42)
n = 5
A = rng.standard_normal((n, n))
b = rng.standard_normal(n)

decomp = decomposition_lr(A, StrategiePivot.PARTIEL)
b_perm = b[decomp.p]
y = substitution_avant(decomp.L, b_perm)
x = substitution_arriere(decomp.R, y)

print(f'x (mine)  = {x}')
print(f'x (numpy) = {np.linalg.solve(A, b)}')
print(f'Écart : {np.linalg.norm(x - np.linalg.solve(A, b), np.inf):.2e}')

## 5. Déterminant comme « Abfallprodukt » (section 2.3.3)

$$\det(A) = (-1)^{\text{nombre de permutations}} \prod_{i=1}^n r_{ii}$$

Coût : $O(n^3/3)$ — bien moins que la formule de Leibniz en $O(n!)$.

Sur une matrice 10×10, Leibniz exigerait $10! = 3{,}6 \cdot 10^6$ termes. La décomposition LR : ~330 opérations.

In [ ]:
rng = np.random.default_rng(0)
for n in [3, 5, 10, 50]:
    A = rng.standard_normal((n, n))
    print(f'n={n:>3}  det(mine) = {determinant(A):>14.4f}  det(numpy) = {np.linalg.det(A):>14.4f}')

## 6. Section 2.3.6 — borne d'erreur et conditionnement

Le script démontre la borne :

$$\frac{\|x_{calc} - x\|}{\|x\|} \leq \kappa(A) \cdot \frac{\|r\|}{\|b\|}$$

où $\kappa(A) = \|A\| \cdot \|A^{-1}\|$ est le **nombre de conditionnement** de $A$. En pratique, avec un solveur stable, $\|r\|/\|b\|$ reste de l'ordre de $\varepsilon_{mach}$, donc :

$$\text{erreur relative} \sim \kappa(A) \cdot \varepsilon_{mach}$$

**Test :** matrices de Hilbert $H_n$ avec $H_{ij} = 1/(i+j-1)$. Leur conditionnement croît exponentiellement avec $n$, donc on perd de la précision à chaque ligne ajoutée.

In [ ]:
print(f'{"n":>3} | {"κ_∞(H_n)":>14} | {"erreur relative":>16}')
print('-' * 42)
for n in [3, 5, 8, 10, 12]:
    H = matrice_hilbert(n)
    x_exact = np.ones(n)
    b = H @ x_exact
    x = resoudre(H, b, StrategiePivot.PARTIEL)
    print(f'{n:>3} | {np.linalg.cond(H, np.inf):>14.2e} | {erreur_relative(x, x_exact):>16.2e}')

In [ ]:
tracer_erreur_vs_conditionnement()
plt.show()

L'erreur observée suit fidèlement la borne théorique $\kappa(A) \cdot \varepsilon_{mach}$. C'est rassurant : notre implémentation est **stable**.

**Règle pratique :** si $\kappa(A) \sim 10^k$, on perd environ $k$ chiffres de précision. À $\kappa = 10^{16}$, on a perdu **toute** la précision en double.

## Récapitulatif (Satz 2.20)

| Opération | Coût |
|---|---|
| Décomposition LR | $\frac{1}{3} n^3$ |
| Substitution avant | $\frac{1}{2} n^2$ |
| Substitution arrière | $\frac{1}{2} n^2$ |
| **Total Ax = b** | $\frac{1}{3} n^3 + n^2$ |
| Pivot partiel | gratuit en ordre dominant |
| Pivot total | + $O(n^3)$, rarement utilisé |

**Le pivot partiel est devenu le standard** : il a la même complexité asymptotique que sans pivot, mais résout les problèmes de stabilité dans la quasi-totalité des cas pratiques.

**Prochain programme :** `jacobi_gauss_seidel.py` — méthodes itératives, qui deviennent indispensables quand $n$ est grand (matrices creuses, équations aux dérivées partielles).